# 📊 Làm sạch Dữ liệu Rise in AI Commits
### Dựa theo cấu trúc Chapter 3 — Data Quality and Processing

---

## 🎯 Mục tiêu
Làm sạch dataset về các AI commits chuẩn bị cho mô hình NLP và báo cáo EDA.

1. **Đánh giá tỷ lệ khuyết:** Phân tích tỷ lệ phần trăm dữ liệu thiếu.
2. **Xử lý lỗi 404 (~3.1%):** Lọc bỏ các dòng thiếu `commit_date` do URL bị chết/chuyển private.
3. **Tái cấu trúc Feature (~63.1%):** Xử lý `change_commit_date` bằng cách tạo Feature `is_modified`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    import missingno as msno
    HAS_MSNO = True
except ImportError:
    HAS_MSNO = False

DATA_PATH = r"data\interim\standardized_data_2026-09-22 23_41_54.csv"
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"✅ Đã tải dữ liệu: {df_raw.shape[0]:,} hàng × {df_raw.shape[1]} cột")

---
# BLOCK 1 — Khảo sát tỷ lệ Missing
---

In [ ]:
missing_stats = pd.DataFrame({
    'Số dòng thiếu': df_raw.isnull().sum(),
    'Tỷ lệ (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
missing_stats = missing_stats[missing_stats['Số dòng thiếu'] > 0].sort_values(by='Tỷ lệ (%)', ascending=False)
print("THỐNG KÊ DỮ LIỆU BỊ THIẾU:")
display(missing_stats)

---
# BLOCK 2 — Handling Missing Data
---
## 2.1 Xóa các commit bị lỗi 404 (~3.1%)
Thống kê thực tế cho thấy tỷ lệ thiếu `commit_date` chỉ là 3.07% (1,082 dòng), KHÔNG PHẢI 8% như ước tính ban đầu. Các dòng này tương ứng với các URL GitHub trả về lỗi 404 (đã bị xóa/private). Việc Drop nhóm này là an toàn và cần thiết để đảm bảo tính chuỗi thời gian (time-series) cho EDA.

In [ ]:
# Thực hiện Drop
df_clean = df_raw.dropna(subset=['commit_date']).copy()
print(f"Dataset sau khi Drop 1,082 dòng lỗi 404: {len(df_clean):,} hàng")

## 2.2 Tái cấu trúc (Feature Engineering) cho cột `change_commit_date` (~63.1%)
Hơn 63% AI commits không có `change_commit_date`. Nguyên nhân: **Code AI sinh ra được dùng ngay mà không hề bị chỉnh sửa (Not Modified)**. Chúng ta sẽ biến sự thiếu hụt này thành đặc trưng (Feature) `is_modified`.

In [ ]:
if 'change_commit_date' in df_clean.columns:
    # Tạo cột is_modified (True nếu có giá trị, False nếu NaN)
    df_clean['is_modified'] = df_clean['change_commit_date'].notna()
    
    # Fill NaN bằng nhãn cụ thể
    df_clean['change_commit_date'] = df_clean['change_commit_date'].fillna('Not Modified')
    if 'change_commit_url' in df_clean.columns:
        df_clean['change_commit_url'] = df_clean['change_commit_url'].fillna('No URL')

    print("Phân phối của các đoạn code AI (Đã bị sửa vs Chưa bị sửa):")
    dist = df_clean['is_modified'].value_counts(normalize=True).mul(100).round(2)
    for val, pct in dist.items():
        label = "Đã bị sửa bởi người (True)" if val else "Chưa bị sửa (False)"
        print(f" - {label}: {pct}%")

---
# BLOCK 3 — Wrap-up
---
Lưu tập dữ liệu đã sạch (Cleaned) để phục vụ cho các báo cáo tiếp theo.

In [ ]:
import os
save_dir = r"data\processed"
os.makedirs(save_dir, exist_ok=True)
CLEAN_DATA_PATH = os.path.join(save_dir, "cleaned_commits_data_final.csv")

df_clean.to_csv(CLEAN_DATA_PATH, index=False)
print(f"✅ Đã lưu dữ liệu sạch vào: {CLEAN_DATA_PATH}")
print(f"Shape cuối cùng: {df_clean.shape[0]:,} hàng × {df_clean.shape[1]} cột")